# Circuit Training trên Google Colab

Chạy Reinforcement Learning cho Macro Placement trên Google Colab (Free GPU)

**Lưu ý:** Chọn Runtime → Change runtime type → GPU (T4)

## Bước 1: Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

## Bước 2: Mount Google Drive (để lưu kết quả)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo folder cho project
!mkdir -p /content/drive/MyDrive/circuit_training_project

## Bước 3: Clone Repositories

In [ ]:
%cd /content

# Clone circuit_training
!git clone https://github.com/google-research/circuit_training.git

# Clone MacroPlacement (cho benchmarks)
!git clone https://github.com/TILOS-AI-Institute/MacroPlacement.git

# Clone DREAMPlace (cho placement engine)
!git clone --recursive https://github.com/limbo018/DREAMPlace.git

## Bước 4: Cài đặt Dependencies

In [ ]:
# Cài TensorFlow và TF-Agents
!pip install -q tensorflow==2.15.0
!pip install -q tf-agents[reverb]==0.19.0
!pip install -q gin-config

# Cài các dependencies khác
!pip install -q gym==0.23.0
!pip install -q timeout-decorator
!pip install -q tensorflow-probability==0.23.0

print('\n✅ Dependencies installed!')

## Bước 5: Build và Cài DREAMPlace

In [ ]:
# Build DREAMPlace (cần CMake)
!apt-get update -qq
!apt-get install -y -qq cmake

%cd /content/DREAMPlace

# Build
!mkdir -p build && cd build && \
cmake .. -DPYTHON_EXECUTABLE=$(which python3) -DCMAKE_BUILD_TYPE=Release && \
make -j4

# Cài đặt
%cd /content/DREAMPlace
!pip install -e .

print('\n✅ DREAMPlace built and installed!')

## Bước 6: Test Import Circuit Training

In [ ]:
import sys
sys.path.insert(0, '/content/circuit_training')

# Test import
from circuit_training.environment.environment import CircuitEnv
print('✅ CircuitEnv imported successfully!')

# Check test data
import os
test_data_path = '/content/circuit_training/circuit_training/environment/test_data'
if os.path.exists(test_data_path):
    print(f'\n✅ Test data available at: {test_data_path}')
    subdirs = [d for d in os.listdir(test_data_path) if os.path.isdir(os.path.join(test_data_path, d))]
    print(f'Available test cases: {subdirs}')

## Bước 7: Tạo Config và Training Script

In [ ]:
# Tạo config file
config_content = '''
import circuit_training.environment.environment
import circuit_training.learning.ppo_main

# Environment
CircuitEnv.netlist_file = '/content/circuit_training/circuit_training/environment/test_data/ariane/netlist.pb.txt'
CircuitEnv.init_placement = '/content/circuit_training/circuit_training/environment/test_data/ariane/initial.plc'

# Training
train.root_dir = '/content/drive/MyDrive/circuit_training_project/logs'
train.num_iterations = 100
train.train_episodes = 100
train.eval_episodes = 10
'''

with open('/content/config.gin', 'w') as f:
    f.write(config_content)

print('✅ Config created at /content/config.gin')

## Bước 8: Chạy Training (PPO)

In [ ]:
%cd /content/circuit_training

# Chạy training với toy benchmark (nhanh hơn)
!python -m circuit_training.learning.ppo_main \
  --root_dir=/content/drive/MyDrive/circuit_training_project/logs \
  --netlist_file=/content/circuit_training/circuit_training/environment/test_data/toy_macro_stdcell/netlist.pb.txt \
  --init_placement=/content/circuit_training/circuit_training/environment/test_data/toy_macro_stdcell/initial.plc \
  --train_episodes=50 \
  --num_iterations=20

## Bước 9: Theo dõi Training với TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=/content/drive/MyDrive/circuit_training_project/logs

## Bước 10: Export và Đánh giá Placement

In [ ]:
# Copy placement output
!cp -r /content/drive/MyDrive/circuit_training_project/logs/* /content/placement_output/

# Liệt kê files
!ls -la /content/drive/MyDrive/circuit_training_project/logs/

## Bước 11: Visualize Placement (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Đọc placement file (nếu có format đơn giản)
# Đây là ví dụ visualization đơn giản

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_xlim(0, 1000)
ax.set_ylim(0, 1000)
ax.set_title('Macro Placement Visualization')
ax.grid(True)
plt.show()

## Tips và Lưu ý

### 1. Tránh mất kết nối Colab:
- Chạy cell này để giữ session alive:
```python
import time
while True:
    time.sleep(60)  # Keep alive
```

### 2. Lưu checkpoint thường xuyên:
- Training logs tự động lưu vào Google Drive
- Có thể resume từ checkpoint

### 3. Benchmark nhỏ trước:
- `toy_macro_stdcell`: ~10 macros (test nhanh)
- `ariane`: ~133 macros (training thật)
- `mempool`: ~200+ macros (lâu hơn)

### 4. Giới hạn Colab:
- Free GPU: T4 hoặc K80 (12 giờ liên tục max)
- RAM: ~12GB
- Disk: ~68GB